In [1]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint
import optuna

In [2]:
import pandas as pd

train = pd.read_csv("train_FE1.csv")

In [3]:
# Pisahkan fitur dan target
X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

# Split internal: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [4]:
# Kolom kategorikal dan numerik
# List kriteria kolom yang dikecualikan (bukan fitur)
exclude_cols = ["id", "addicted_label"]

# 1. Definisi Manual untuk Categorical Features
categorical_features = [
    "gender",
    "stress_level",
    "academic_work_impact",
    "sleep_deprived_flag",
    "weekend_binge_flag",
    "screen_risk_bin",
    "sleep_risk_bin",
    "notif_risk_bin",
    "extreme_addiction_pattern",
    "compulsive_usage_flag"
]

# 2. Loop Otomatis untuk Numeric Features
numeric_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time",
    "screen_time_gap",
    "weekend_screen_ratio",
    "social_media_ratio",
    "gaming_ratio",
    "notifications_per_screen_hour",
    "screen_to_sleep_ratio",
    "total_active_screen_hours",
    "non_sleep_screen_share",
    "app_opens_per_screen_hour",
    "notif_to_open_ratio",
    "leisure_screen_hours",
    "leisure_ratio",
    "unaccounted_screen_time",
    "stress_screen_interaction",
    "dominant_activity_ratio",
    "activity_entropy",
    "productive_hours",
    "leisure_vs_productive_gap",
    "leisure_to_productive_ratio",
    "social_vs_gaming_gap",
    "social_to_gaming_ratio",
    "screen_sleep_gap",
    "screen_sleep_product",
    "weekend_binge_gap",
    "notif_sleep_pressure",
    "opens_sleep_pressure",
    "stress_sleep_interaction",
    "stress_leisure_interaction",
    "stress_notif_interaction",
    "opens_per_leisure_hour",
    "notif_per_leisure_hour",
    "notif_open_gap",
    "notif_open_abs_gap",
    "unaccounted_screen_ratio",
    "unaccounted_abs",
    "accounted_screen_hours",
    "accounted_screen_ratio"
] 

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ]
)

# 10-fold CV
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [5]:
# Pipeline baru khusus untuk tuning
tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(
        random_state=42,
        early_stopping=True
    ))
])

In [6]:
def objective(trial):
    # Parameter yang dicari Optuna
    params = {
        "model__learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "model__max_iter": trial.suggest_int(
            "max_iter", 200, 1500
        ),
        "model__max_leaf_nodes": trial.suggest_int(
            "max_leaf_nodes", 15, 100
        ),
        "model__max_depth": trial.suggest_int(
            "max_depth", 3, 15
        ),
        "model__min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 10, 100
        ),
        "model__l2_regularization": trial.suggest_float(
            "l2_regularization", 1e-4, 10.0, log=True
        ),
        "model__max_bins": trial.suggest_int(
            "max_bins", 63, 255
        ),
        "model__max_features": trial.suggest_float(
            "max_features", 0.5, 1.0
        )
    }

    # Salin pipeline agar pipeline asli tidak berubah
    pipeline = tuning_pipeline.set_params(**params)

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1
    )

    return scores.mean()

In [7]:
study = optuna.create_study(
    direction="maximize",
    study_name="hgb_roc_auc"
)

study.optimize(
    objective,
    n_trials=1 00,
    show_progress_bar=True
)

print("Best CV ROC-AUC:", study.best_value)
print("Best parameters:")
print(study.best_params)

[I 2026-08-29 19:01:03,297] A new study created in memory with name: hgb_roc_auc


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-08-29 19:13:02,912] Trial 0 finished with value: 0.9544824618872141 and parameters: {'learning_rate': 0.1189748874289051, 'max_iter': 980, 'max_leaf_nodes': 34, 'max_depth': 4, 'min_samples_leaf': 44, 'l2_regularization': 0.08283096817427034, 'max_bins': 90, 'max_features': 0.9507603138050215}. Best is trial 0 with value: 0.9544824618872141.
[W 2026-08-29 19:16:19,025] Trial 1 failed with parameters: {'learning_rate': 0.01736818361966887, 'max_iter': 1361, 'max_leaf_nodes': 46, 'max_depth': 4, 'min_samples_leaf': 31, 'l2_regularization': 0.0009271546450195954, 'max_bins': 85, 'max_features': 0.7251193377411221} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\danis\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\danis\AppData\Local\Temp\ipykernel_21000\252448507.py", line 33, in objective
    scores = cross_val_

KeyboardInterrupt: 

In [ ]:
best_params = {
    f"model__{key}": value
    for key, value in study.best_params.items()
}

best_pipeline = tuning_pipeline.set_params(**best_params)

best_pipeline.fit(
    X_train,
    y_train
)

print("Model terbaik selesai dilatih.")

In [ ]:
from sklearn.metrics import roc_auc_score

y_test_probability = best_pipeline.predict_proba(X_test)[:, 1]

test_auc = roc_auc_score(
    y_test,
    y_test_probability
)

print(f"Holdout ROC-AUC: {test_auc:.6f}")

In [ ]:
# import json
# import joblib

# # Simpan parameter terbaik
# with open("hgb_optuna_best_params.json", "w") as file:
#     json.dump(study.best_params, file, indent=4)

# # Simpan pipeline terlatih
# joblib.dump(
#     best_pipeline,
#     "hgb_optuna_best.joblib"
# )

# print("Parameter dan model berhasil disimpan.")